## Deployment: Aplikasi Prediksi Harga Cabai Merah Besar dengan Gradio

Notebook ini merupakan tahap **Deployment** dalam metodologi **CRISP-DM**, yang bertujuan untuk membangun antarmuka web interaktif menggunakan **Gradio** sehingga pengguna non-teknis dapat melakukan prediksi harga cabai secara real-time.

Model yang digunakan adalah **Linear Regression** terbaik yang telah disimpan pada tahap evaluasi sebelumnya (`model_cabai_lr.pkl`).

Antarmuka dilengkapi dengan **tombol preset** untuk skenario cuaca (kemarau, hujan, transisi) dan skenario harga historis (rendah, normal, tinggi, sangat tinggi), yang nilainya dihitung dari dataset aktual sehingga pengguna tidak perlu mengisi semua field secara manual.

### 1. Instalasi Dependency

Pastikan pustaka `gradio` telah terinstal. Jika belum, jalankan cell berikut.

In [1]:
# !pip install gradio joblib scikit-learn pandas numpy

### 2. Import Library

In [2]:
import os
import joblib
import pandas as pd
import numpy as np
import gradio as gr
from pathlib import Path

### 3. Memuat Model Linear Regression

Model terbaik (`model_cabai_lr.pkl`) dimuat dari folder `models/`. Path dibuat fleksibel agar notebook dapat dijalankan dari berbagai lokasi.

In [3]:
# Path fleksibel untuk mencari file model
possible_model_paths = [
    Path('../models/model_cabai_lr.pkl'),
    Path('model_cabai_lr.pkl'),
    Path('models/model_cabai_lr.pkl'),
]

model_path = None
for path in possible_model_paths:
    if path.exists():
        model_path = path
        print(f"Model ditemukan di: {path}")
        break

if model_path is None:
    raise FileNotFoundError("File model_cabai_lr.pkl tidak ditemukan. Pastikan file berada di folder models/.")

model = joblib.load(model_path)
print(f"Model berhasil dimuat: {type(model).__name__}")

Model ditemukan di: ..\models\model_cabai_lr.pkl
Model berhasil dimuat: LinearRegression


c:\Users\jeebr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.8.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### 4. Pemetaan Bulan dan Fungsi Prediksi

Pada antarmuka Gradio, bulan ditampilkan dalam format nama bulan Indonesia (Januari–Desember), lalu dikonversi ke integer (1–12) sebelum dikirim ke model.

Fungsi `prediksi_harga_cabai` menerima 6 input dari UI, menyusunnya ke DataFrame dengan kolom yang sesuai dengan `X_train`, melakukan prediksi, dan memformat output ke Rupiah.

In [4]:
# Pemetaan nama bulan Indonesia ke integer
BULAN_MAP = {
    "Januari": 1, "Februari": 2, "Maret": 3, "April": 4, "Mei": 5, "Juni": 6,
    "Juli": 7, "Agustus": 8, "September": 9, "Oktober": 10, "November": 11, "Desember": 12
}

# Urutan kolom fitur yang persis sama dengan X_train
FEATURE_COLUMNS = ['Cabai_lag_1', 'Cabai_lag_7', 'RR_lag_45', 'RH_lag_30', 'RR_rolling_mean_14', 'bulan']


def prediksi_harga_cabai(cabai_lag_1, cabai_lag_7, rr_lag_45, rh_lag_30, rr_rolling_mean_14, nama_bulan):
    try:
        # Konversi bulan string ke integer
        bulan_val = BULAN_MAP.get(nama_bulan, 1)

        # Susun data ke DataFrame dengan urutan kolom yang sesuai X_train
        input_data = pd.DataFrame({
            'Cabai_lag_1': [float(cabai_lag_1)],
            'Cabai_lag_7': [float(cabai_lag_7)],
            'RR_lag_45': [float(rr_lag_45)],
            'RH_lag_30': [float(rh_lag_30)],
            'RR_rolling_mean_14': [float(rr_rolling_mean_14)],
            'bulan': [int(bulan_val)]
        })

        # Prediksi menggunakan model Linear Regression
        predicted_value = model.predict(input_data)[0]

        # Batasi agar harga tidak bernilai negatif
        predicted_value = max(0.0, predicted_value)

        # Format output ke Rupiah
        return f"Rp {predicted_value:,.2f}"

    except Exception as err:
        return f"Terjadi kesalahan saat memproses data: {str(err)}"

### 4.1. Preset Skenario Cuaca dan Harga Historis

Tombol preset memungkinkan pengguna mengisi seluruh field input secara otomatis hanya dengan satu klik. Nilai preset dihitung dari dataset aktual (`dataset_merge_cleaning.csv`) berdasarkan statistik deskriptif dan kondisi musim yang teridentifikasi dalam data:

- **Preset Cuaca**: mewakili kondisi kemarau panas, kemarau sejuk, musim transisi, hujan ringan, dan hujan lebat di Bandung.
- **Preset Harga**: mewakili harga cabai pada level rendah (Q10), normal (median), tinggi (Q75), dan sangat tinggi (Q90+).

Ketika tombol preset diklik, seluruh field input (harga historis, variabel cuaca, dan bulan) akan terisi otomatis sesuai skenario yang dipilih.

In [5]:
# --- PRESET CUACA BANDUNG ---
# Nilai dihitung dari analisis statistik data historis BMKG Bandung dalam dataset merge.
# RR_lag_45 = curah hujan harian 45 hari lalu (mm)
# RH_lag_30 = kelembapan rata-rata 30 hari lalu (%)
# RR_rolling_mean_14 = rerata curah hujan 14 hari terakhir (mm)

PRESET_CUACA = {
    "☀️ Kemarau Panas": {
        "rr_lag_45": 0.0,
        "rh_lag_30": 65,
        "rr_rolling_mean_14": 0.0,
        "nama_bulan": "September",
    },
    "🌤️ Kemarau Sejuk": {
        "rr_lag_45": 0.0,
        "rh_lag_30": 72,
        "rr_rolling_mean_14": 1.5,
        "nama_bulan": "Agustus",
    },
    "🌥️ Musim Transisi": {
        "rr_lag_45": 3.0,
        "rh_lag_30": 78,
        "rr_rolling_mean_14": 5.0,
        "nama_bulan": "Oktober",
    },
    "🌧️ Hujan Ringan": {
        "rr_lag_45": 8.0,
        "rh_lag_30": 82,
        "rr_rolling_mean_14": 7.5,
        "nama_bulan": "November",
    },
    "⛈️ Hujan Lebat": {
        "rr_lag_45": 35.0,
        "rh_lag_30": 88,
        "rr_rolling_mean_14": 15.0,
        "nama_bulan": "Januari",
    },
}

# --- PRESET HARGA CABAI HISTORIS ---
# Nilai dihitung dari quantile dataset PIHPS Jakarta.
# Cabai_lag_1 = harga 1 hari sebelumnya (Q10/25/50/75/90)
# Cabai_lag_7 = harga 7 hari sebelumnya

PRESET_HARGA = {
    "📉 Harga Rendah": {
        "cabai_lag_1": 42500,
        "cabai_lag_7": 42500,
        "nama_bulan": "November",
    },
    "📊 Harga Normal": {
        "cabai_lag_1": 53500,
        "cabai_lag_7": 53500,
        "nama_bulan": "September",
    },
    "📈 Harga Tinggi": {
        "cabai_lag_1": 65500,
        "cabai_lag_7": 65550,
        "nama_bulan": "Juni",
    },
    "🔥 Harga Sangat Tinggi": {
        "cabai_lag_1": 71400,
        "cabai_lag_7": 71400,
        "nama_bulan": "Maret",
    },
}

# Fungsi handler untuk tombol preset cuaca
def preset_cuaca_handler(skenario):
    data = PRESET_CUACA[skenario]
    return data["rr_lag_45"], data["rh_lag_30"], data["rr_rolling_mean_14"], data["nama_bulan"]

# Fungsi handler untuk tombol preset harga
def preset_harga_handler(skenario):
    data = PRESET_HARGA[skenario]
    return data["cabai_lag_1"], data["cabai_lag_7"], data["nama_bulan"]

print("Preset cuaca:")
for k, v in PRESET_CUACA.items():
    print(f"  {k}: RR_lag_45={v['rr_lag_45']}, RH_lag_30={v['rh_lag_30']}, RR_roll={v['rr_rolling_mean_14']}, Bulan={v['nama_bulan']}")

print("\nPreset harga:")
for k, v in PRESET_HARGA.items():
    print(f"  {k}: Cabai_lag_1={v['cabai_lag_1']}, Cabai_lag_7={v['cabai_lag_7']}, Bulan={v['nama_bulan']}")

Preset cuaca:
  ☀️ Kemarau Panas: RR_lag_45=0.0, RH_lag_30=65, RR_roll=0.0, Bulan=September
  🌤️ Kemarau Sejuk: RR_lag_45=0.0, RH_lag_30=72, RR_roll=1.5, Bulan=Agustus
  🌥️ Musim Transisi: RR_lag_45=3.0, RH_lag_30=78, RR_roll=5.0, Bulan=Oktober
  🌧️ Hujan Ringan: RR_lag_45=8.0, RH_lag_30=82, RR_roll=7.5, Bulan=November
  ⛈️ Hujan Lebat: RR_lag_45=35.0, RH_lag_30=88, RR_roll=15.0, Bulan=Januari

Preset harga:
  📉 Harga Rendah: Cabai_lag_1=42500, Cabai_lag_7=42500, Bulan=November
  📊 Harga Normal: Cabai_lag_1=53500, Cabai_lag_7=53500, Bulan=September
  📈 Harga Tinggi: Cabai_lag_1=65500, Cabai_lag_7=65550, Bulan=Juni
  🔥 Harga Sangat Tinggi: Cabai_lag_1=71400, Cabai_lag_7=71400, Bulan=Maret


### 5. Pengujan Fungsi Prediksi

Sebelum membangun UI, kita uji fungsi prediksi dengan data contoh untuk memastikan model dan fungsi bekerja dengan benar.

In [6]:
# Uji fungsi prediksi dengan data contoh (sama seperti pada notebook modelling)
hasil_test = prediksi_harga_cabai(
    cabai_lag_1=70000,
    cabai_lag_7=68000,
    rr_lag_45=10,
    rh_lag_30=82,
    rr_rolling_mean_14=7,
    nama_bulan="Juni"
)

print(f"Hasil prediksi uji coba: {hasil_test}")

Hasil prediksi uji coba: Rp 69,890.85


### 6. Desain Antarmuka Gradio

Antarmuka Gradio dirancang dengan tema **Soft** untuk tampilan yang bersih dan profesional. Layout terdiri dari:
- **Baris preset harga**: tombol untuk mengisi skenario harga cabai historis secara otomatis
- **Baris preset cuaca**: tombol untuk mengisi skenario iklim Bandung secara otomatis
- **Kolom kiri**: Parameter harga cabai historis (Harga Kemarin, Harga Seminggu Lalu, Bulan Prediksi)
- **Kolom kanan**: Parameter iklim Bandung (Curah Hujan Lag-45, Kelembapan Lag-30, Rerata Curah Hujan 14 Hari)
- **Baris bawah**: Tombol prediksi dan output hasil prediksi

Ketika tombol preset diklik, seluruh field input yang relevan akan terisi otomatis sesuai skenario yang dipilih, sehingga pengguna tidak perlu mengisi setiap field secara manual.

In [7]:
with gr.Blocks(title="Prediksi Harga Cabai DKI Jakarta") as demo:
    gr.Markdown(
        """
        # 🌶️ Aplikasi Prediksi Harga Cabai Merah Besar Jakarta
        **Proyek Akhir Data Mining (DM210) - STT Terpadu Nurul Fikri**

        Aplikasi ini memprediksi harga harian Cabai Merah Besar di DKI Jakarta menggunakan model **Linear Regression** terbaik yang telah dilatih menggunakan data historis harga cabai dan data iklim/cuaca BMKG Bandung.
        """
    )

    # --- BARIS PRESET HARGA ---
    gr.Markdown("### 💰 Preset Skenario Harga Historis")
    gr.Markdown("*Klik salah satu tombol untuk mengisi harga cabai historis dan bulan secara otomatis.*")
    with gr.Row():
        btn_harga_rendah = gr.Button("📉 Rendah (Q10)", size="sm")
        btn_harga_normal = gr.Button("📊 Normal (Median)", size="sm")
        btn_harga_tinggi = gr.Button("📈 Tinggi (Q75)", size="sm")
        btn_harga_sangat_tinggi = gr.Button("🔥 Sangat Tinggi (Q90)", size="sm", variant="secondary")

    # --- BARIS PRESET CUACA ---
    gr.Markdown("### 🌦️ Preset Skenario Iklim Bandung")
    gr.Markdown("*Klik salah satu tombol untuk mengisi variabel cuaca Bandung dan bulan secara otomatis.*")
    with gr.Row():
        btn_cuaca_kemarau_panas = gr.Button("☀️ Kemarau Panas", size="sm")
        btn_cuaca_kemarau_sejuk = gr.Button("🌤️ Kemarau Sejuk", size="sm")
        btn_cuaca_transisi = gr.Button("🌥️ Transisi", size="sm")
        btn_cuaca_hujan_ringan = gr.Button("🌧️ Hujan Ringan", size="sm")
        btn_cuaca_hujan_lebat = gr.Button("⛈️ Hujan Lebat", size="sm", variant="secondary")

    # --- INPUT FORM ---
    with gr.Row():
        # Kolom Input Kiri (Harga Historis)
        with gr.Column():
            gr.Markdown("### 💵 Parameter Harga Cabai Historis")
            cabai_lag_1 = gr.Number(label="Harga Cabai Kemarin (Rp/Kg)", value=64600)
            cabai_lag_7 = gr.Number(label="Harga Cabai Seminggu Lalu (Rp/Kg)", value=57000)
            nama_bulan = gr.Dropdown(
                label="Bulan Prediksi",
                choices=list(BULAN_MAP.keys()),
                value="April"
            )

        # Kolom Input Kanan (Variabel Cuaca Bandung)
        with gr.Column():
            gr.Markdown("### 🌦️ Parameter Iklim Bandung")
            rr_lag_45 = gr.Number(label="Curah Hujan Bandung Lag-45 Hari (mm)", value=0.0)
            rh_lag_30 = gr.Slider(label="Kelembapan Bandung Lag-30 Hari (%)", minimum=0, maximum=100, step=1, value=78)
            rr_rolling_mean_14 = gr.Number(label="Rata-rata Curah Hujan Bandung 14 Hari Terakhir (mm)", value=11.36)

    # --- OUTPUT & TOMBOL PREDIKSI ---
    with gr.Row():
        with gr.Column(scale=1):
            btn_predict = gr.Button("🔮 Hitung Estimasi Harga", variant="primary")
        with gr.Column(scale=2):
            output_text = gr.Textbox(
                label="Hasil Prediksi Harga Cabai Merah Besar (Rupiah/Kg)",
                interactive=False,
                placeholder="Hasil prediksi akan muncul di sini..."
            )

    # --- HUBUNGKAN TOMBOL PREDIKSI ---
    btn_predict.click(
        fn=prediksi_harga_cabai,
        inputs=[cabai_lag_1, cabai_lag_7, rr_lag_45, rh_lag_30, rr_rolling_mean_14, nama_bulan],
        outputs=output_text
    )

    # --- HUBUNGKAN TOMBOL PRESET HARGA ---
    # Preset harga mengisi: cabai_lag_1, cabai_lag_7, nama_bulan
    harga_outputs = [cabai_lag_1, cabai_lag_7, nama_bulan]

    btn_harga_rendah.click(fn=lambda: preset_harga_handler("📉 Harga Rendah"), outputs=harga_outputs)
    btn_harga_normal.click(fn=lambda: preset_harga_handler("📊 Harga Normal"), outputs=harga_outputs)
    btn_harga_tinggi.click(fn=lambda: preset_harga_handler("📈 Harga Tinggi"), outputs=harga_outputs)
    btn_harga_sangat_tinggi.click(fn=lambda: preset_harga_handler("🔥 Harga Sangat Tinggi"), outputs=harga_outputs)

    # --- HUBUNGKAN TOMBOL PRESET CUACA ---
    # Preset cuaca mengisi: rr_lag_45, rh_lag_30, rr_rolling_mean_14, nama_bulan
    cuaca_outputs = [rr_lag_45, rh_lag_30, rr_rolling_mean_14, nama_bulan]

    btn_cuaca_kemarau_panas.click(fn=lambda: preset_cuaca_handler("☀️ Kemarau Panas"), outputs=cuaca_outputs)
    btn_cuaca_kemarau_sejuk.click(fn=lambda: preset_cuaca_handler("🌤️ Kemarau Sejuk"), outputs=cuaca_outputs)
    btn_cuaca_transisi.click(fn=lambda: preset_cuaca_handler("🌥️ Musim Transisi"), outputs=cuaca_outputs)
    btn_cuaca_hujan_ringan.click(fn=lambda: preset_cuaca_handler("🌧️ Hujan Ringan"), outputs=cuaca_outputs)
    btn_cuaca_hujan_lebat.click(fn=lambda: preset_cuaca_handler("⛈️ Hujan Lebat"), outputs=cuaca_outputs)

    gr.Markdown(
        """
        ---
        *Aplikasi dikembangkan sebagai bagian dari rencana deployment final projek data mining.*
        """
    )

print("Antarmuka Gradio berhasil dibangun.")

Antarmuka Gradio berhasil dibangun.


### 7. Jalankan Aplikasi Gradio

Aplikasi akan dijalankan secara lokal pada `http://127.0.0.1:7860`. Jika ingin membagikan link publik sementara (aktif 72 jam), ubah parameter `share=True` pada `demo.launch()`.

> **Catatan Gradio 6.x**: Parameter `theme` telah dipindahkan dari `gr.Blocks()` ke `demo.launch()` pada Gradio versi 6.0+.

In [8]:
# Jalankan aplikasi Gradio
# share=True untuk membagikan link publik sementara (berlaku 72 jam)
demo.launch(theme=gr.themes.Soft(), server_name="127.0.0.1", server_port=7860, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
